In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l6.5 FlashAttention

> 🎯 **Goal:** Understand how FlashAttention computes the exact same attention result
while never storing the giant score matrix, then verify the outputs match.

**Why this matters.** The attention we built so far forms the full `T x T` matrix of
scores (every token against every token). At long sequence lengths that matrix is
enormous and dominates both memory and the slow trips to GPU memory. FlashAttention
is the kernel that made long-context training practical; it is the default attention
path in every serious framework today.

**The intuition.** Imagine multiplying two huge matrices but only having a small desk.
Instead of laying the whole giant result on the desk at once, you bring in one *tile*
at a time, do the partial work, keep a running summary, and move on. FlashAttention
does exactly this: it walks the attention in tiles, maintaining a running softmax
(running max and running sum) so it never has to hold the full score matrix anywhere.
The arithmetic is identical to the textbook version, just reordered to stay in fast
on-chip memory.

**The mechanics.** The cost of attention is not just the multiply-adds; it is moving
the `T x T` scores to and from slow memory. FlashAttention fuses the score, softmax,
and value-weighting steps into one tiled pass that keeps intermediates in fast memory
and uses the online-softmax trick to combine tiles correctly. The result is
mathematically the same output (matching to floating-point tolerance), produced with
far less memory traffic. We taught attention with the explicit `T x T` version because
you can read it; in production you call the fused `F.scaled_dot_product_attention`,
which dispatches to FlashAttention when it can.

In [ ]:
from torch.nn import functional as F
from pocketlm import scaled_dot_product_attention

torch.manual_seed(0)
q = torch.randn(1, 4, 6, 8)
k = torch.randn(1, 4, 6, 8)
v = torch.randn(1, 4, 6, 8)
ours, _ = scaled_dot_product_attention(q, k, v, causal=True)
fused = F.scaled_dot_product_attention(q, k, v, is_causal=True)  # fused kernel
print("max difference:", (ours - fused).abs().max().item())

**What just happened.** The max difference between our explicit attention and PyTorch's
fused kernel is essentially zero (down at floating-point noise). That is the headline:
FlashAttention is not an approximation. It produces the *same* numbers, just computed
with a memory-friendly schedule. So in production you reach for the fused path for the
speed and memory win, while in this course we keep the explicit version because you can
actually read what it does.

⚠️ **Common pitfalls**
- Expecting bit-identical output. Reordered floating-point operations differ in the
  last few bits, so compare with a tolerance (`allclose`), never `==`.
- Forgetting the causal mask. Pass `is_causal=True` to the fused call (as we do here),
  or it will attend to future tokens and quietly break the training objective.
- Assuming FlashAttention changes the math. It does not change the result; it changes
  the *memory access pattern*. The win is fewer trips to slow memory, not different
  outputs.

🏋️ **Try it yourself.** Bump the sequence length from 6 to something larger (say 256)
and rerun. The max difference should stay near zero, but the explicit version now
builds a much bigger score matrix, which is exactly the cost FlashAttention avoids.

In [ ]:
assert torch.allclose(ours, fused, atol=1e-5)